# 07 — Analiza czasowa (5 wariantów bazowych)

Profil **opóźnień** pipeline’u Research API (batch `eval-main`, lokalny LLM `gemma4:e2b`).

**Warianty:** `baseline0` → `baseline1` → `eksperyment1-gin` → `eksperyment1-bm25` → `eksperyment2`

**Źródło:** `output/runs/runs_flat.csv`  
**Wyjście:** `output/latency/` (tabela + wykres faz)

Powiązane: jakość w `03_offline_analysis.ipynb`; pilot szybkiego LLM w `06_openai_selfrag_pilot.ipynb`.


## 0. Konfiguracja


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv

EVAL = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd() / "evaluation"
load_dotenv(EVAL / ".env")
load_dotenv(EVAL / ".env.example")

BATCH_ID = os.getenv("EVAL_BATCH_ID", "eval-main")
RUNS_FLAT = EVAL / "output" / "runs" / "runs_flat.csv"
OUT = EVAL / "output" / "latency"
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

VARIANT_ORDER = [
    "baseline0",
    "baseline1",
    "eksperyment1-gin",
    "eksperyment1-bm25",
    "eksperyment2",
]
LABELS = {
    "baseline0": "W1 Baseline (B0)",
    "baseline1": "W2 Parent-Child (B1)",
    "eksperyment1-gin": "W3 Hybryda GIN",
    "eksperyment1-bm25": "W4 Hybryda BM25",
    "eksperyment2": "W5 Self-RAG",
}
SHORT = {
    "baseline0": "W1",
    "baseline1": "W2",
    "eksperyment1-gin": "W3",
    "eksperyment1-bm25": "W4",
    "eksperyment2": "W5",
}
PHASES = [
    ("rewrite_s", "Query rewrite", "#2E86AB"),
    ("retrieval_s", "Retrieval", "#28A745"),
    ("rerank_s", "Rerank", "#E76F51"),
    ("llm_s", "LLM generate", "#6C5CE7"),
    ("self_rag_s", "Self-RAG", "#F4A261"),
]
AGG = os.getenv("LATENCY_AGG", "median")  # "median" | "mean"

print("BATCH", BATCH_ID)
print("RUNS ", RUNS_FLAT.exists(), RUNS_FLAT)
print("OUT  ", OUT)
print("AGG  ", AGG)


## 1. Załaduj runy


In [ ]:
assert RUNS_FLAT.exists(), f"Brak {RUNS_FLAT} — najpierw notebook 01"

raw = pd.read_csv(RUNS_FLAT)
raw = raw[raw["variant"].isin(VARIANT_ORDER)].copy()

if "api_status" in raw.columns:
    ok = raw[raw["api_status"].astype(str).str.lower().eq("ok")].copy()
elif "ok" in raw.columns:
    ok = raw[raw["ok"] == True].copy()
elif "answer" in raw.columns:
    ok = raw[raw["answer"].notna()].copy()
else:
    ok = raw.copy()

COLMAP = {
    "client_latency_ms": ["client_latency_ms"],
    "timing_total_ms": ["timing_total_ms", "total_ms"],
    "timing_rewrite_ms": ["timing_rewrite_ms"],
    "timing_retrieval_wall_ms": ["timing_retrieval_wall_ms", "timing_retrieval_ms"],
    "timing_rerank_ms": ["timing_rerank_ms"],
    "timing_llm_ms": ["timing_llm_ms"],
    "timing_self_rag_grade_ms": ["timing_self_rag_grade_ms", "timing_self_rag_ms"],
    "retrieval_passes": ["retrieval_passes"],
}

def pick(df, names):
    for n in names:
        if n in df.columns:
            return pd.to_numeric(df[n], errors="coerce")
    return pd.Series(np.nan, index=df.index)

for canon, aliases in COLMAP.items():
    ok[canon] = pick(ok, aliases)

ok["label"] = ok["variant"].map(LABELS)
ok["short"] = ok["variant"].map(SHORT)

print("wiersze OK:", len(ok))
display(ok.groupby("variant").size().reindex(VARIANT_ORDER).rename("n").to_frame())


## 2. Profil faz (tabela)


In [ ]:
def agg_series(s: pd.Series, how: str) -> float:
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean(skipna=True) if how == "mean" else s.median(skipna=True))

rows = []
for v in VARIANT_ORDER:
    g = ok[ok["variant"] == v]
    if g.empty:
        continue
    self_rag = g["timing_self_rag_grade_ms"].fillna(0)
    rows.append(
        {
            "variant": v,
            "label": LABELS[v],
            "short": SHORT[v],
            "n": len(g),
            "e2e_ms": agg_series(g["client_latency_ms"], AGG),
            "e2e_p95_ms": float(g["client_latency_ms"].quantile(0.95)),
            "rewrite_ms": agg_series(g["timing_rewrite_ms"], AGG),
            "retrieval_ms": agg_series(g["timing_retrieval_wall_ms"], AGG),
            "rerank_ms": agg_series(g["timing_rerank_ms"], AGG),
            "llm_ms": agg_series(g["timing_llm_ms"], AGG),
            "self_rag_ms": agg_series(self_rag, AGG),
            "self_rag_retry_rate": float((g["retrieval_passes"] == 2).mean())
            if g["retrieval_passes"].notna().any()
            else 0.0,
        }
    )

profile = pd.DataFrame(rows)
for c in ["e2e", "rewrite", "retrieval", "rerank", "llm", "self_rag"]:
    profile[f"{c}_s"] = profile[f"{c}_ms"] / 1000.0
profile["e2e_p95_s"] = profile["e2e_p95_ms"] / 1000.0
# brak fazy (B0/B1 bez rerank) → 0 w wykresach / deltach
for c in ["retrieval_s", "rerank_s", "self_rag_s"]:
    profile[c] = profile[c].fillna(0.0)

table = pd.DataFrame(
    {
        "Wariant": profile["label"],
        "n": profile["n"],
        f"E2E_{AGG}_s": profile["e2e_s"].round(2),
        "E2E_p95_s": profile["e2e_p95_s"].round(2),
        f"Rewrite_{AGG}_s": profile["rewrite_s"].round(2),
        f"Retrieval_{AGG}_s": profile["retrieval_s"].round(2),
        f"Rerank_{AGG}_s": profile["rerank_s"].round(2),
        f"LLM_{AGG}_s": profile["llm_s"].round(2),
        f"SelfRAG_{AGG}_s": profile["self_rag_s"].round(3),
        "SelfRAG_retry_%": (profile["self_rag_retry_rate"] * 100).round(1),
    }
)
display(table)

profile.to_csv(OUT / "latency_profile.csv", index=False, encoding="utf-8-sig")
table.to_csv(OUT / "latency_table.csv", index=False, encoding="utf-8-sig")
print("Zapisano:", OUT / "latency_table.csv")


## 3. Profil opóźnień — fazy

Stacked bar: mediana czasu poszczególnych faz pipeline’u per wariant.


In [ ]:
labels = profile["short"].tolist()
x = np.arange(len(labels))
bar_w = 0.55
fig, ax = plt.subplots(figsize=(11.5, 6.0), constrained_layout=True)

PHASE_STYLE = [
    ("rewrite_s", "Query rewrite", "#2E86AB"),
    ("retrieval_s", "Retrieval", "#28A745"),
    ("rerank_s", "Rerank", "#E76F51"),
    ("llm_s", "LLM generate", "#6C5CE7"),
    ("self_rag_s", "Self-RAG", "#F4A261"),
]

# zbierz wysokości segmentów do etykiet
seg_vals = {key: profile[key].fillna(0).to_numpy(dtype=float) for key, _, _ in PHASE_STYLE}

bottom = np.zeros(len(labels))
for key, name, color in PHASE_STYLE:
    vals = seg_vals[key]
    ax.bar(
        x,
        vals,
        bottom=bottom,
        label=name,
        color=color,
        width=bar_w,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    # etykiety z prawej strony każdego segmentu
    for i_bar, (xi, h, b) in enumerate(zip(x, vals, bottom)):
        if h < 0.08:
            continue
        mid = b + h / 2
        ax.text(
            xi + bar_w / 2 + 0.07,
            mid,
            f"{h:.1f}",
            ha="left",
            va="center",
            fontsize=8,
            color=color,
            fontweight="medium",
            zorder=4,
        )
    bottom = bottom + vals

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_xlim(-0.55, len(labels) - 0.15)
ax.set_ylabel("Czas [s]")
ax.set_title("Profil opóźnień per wariant w poszczególnych fazach (mediana)")
ax.set_ylim(0, bottom.max() * 1.10)
ax.yaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

leg = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=5,
    frameon=True,
    fontsize=9,
)
leg.get_frame().set_edgecolor("#C8C8C8")
leg.get_frame().set_linewidth(0.8)

path = FIG / f"01_phase_profile_{AGG}.png"
fig.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print("Wykres:", path)
